In [ ]:
# Put the repository root on sys.path, wherever the kernel was started from.
import os, sys

def _find_repo_root():
    _pkgs = ("genotyping", "pipelines", "screening", "sequencing")
    for _cand in [os.getcwd(), *sys.path]:
        if not _cand:
            continue
        _d = os.path.abspath(_cand)
        while True:
            if all(os.path.isdir(os.path.join(_d, _p)) for _p in _pkgs):
                return _d
            _parent = os.path.dirname(_d)
            if _parent == _d:
                break
            _d = _parent
    return None

_repo_root = _find_repo_root()
if _repo_root and _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)



In [11]:
import config
import importlib
importlib.reload(config)
import pandas as pd

samples = pd.read_excel('meta/samples_info.xlsx', index_col=0)

sample_names = samples.index
fasta_files = sample_names.map(config.fastq_file)
sam_files = sample_names.map(config.sam_file)
aligned_files = sample_names.map(config.aligned_file)

In [14]:
import sys
import os
from genotyping import alignment
import importlib
importlib.reload(alignment) # force reload so that we don't have to restart kernel after making changes to the file
from concurrent.futures import ProcessPoolExecutor


In [12]:
with ProcessPoolExecutor() as executor:
    # Submit all the tasks and process them in parallel
    # consume the iterator so a failure in any worker is raised here
    list(executor.map(alignment.run_bwa,
                 fasta_files, [config.ref_genome_file]*len(fasta_files), sam_files))

bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L0.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L8.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L17.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L24.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L16.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L10.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L1.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L2.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L11.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03_TnpB_L25.fastq.gz
bwa mem -t 1 -k 19 ref/TnpB-omega_amplicon.fasta data/fastqs_renamed/A8-03

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::bwa_idx_l

In [15]:
with ProcessPoolExecutor() as executor:
    results = executor.map(alignment.reconstruct_sequences,
                           sam_files, [config.ref_seq]*len(sam_files), aligned_files, sample_names)
results = list(results)

In [17]:
import pandas as pd
df_res = pd.DataFrame(results)
df_res['mapped_reads_frac'] = df_res.mapped_reads / df_res.total_reads
df_res.to_excel('results/alignment_stats.xlsx')
df_res

,sample,total_reads,mapped_reads,full_reads,num_reads_with_indels,num_indels,mapped_reads_frac
0,A8-03_TnpB_L0,2233,2183,11,6,16,0.977609
1,A8-03_TnpB_L8,2826,2681,1784,584,1613,0.948691
2,A8-03_TnpB_L16,3358,3191,2517,782,2355,0.950268
3,A8-03_TnpB_L24,1204,879,452,52,74,0.730066
4,A8-03_TnpB_L1,2228,1946,12,7,11,0.873429
5,A8-03_TnpB_L17,3621,2696,1689,844,2206,0.744546
6,A8-03_TnpB_L25,701,593,310,38,62,0.845934
7,A8-03_TnpB_L2,2498,1964,125,41,142,0.786229
8,A8-03_TnpB_L10,1354,1302,537,151,450,0.961595
9,A8-03_TnpB_L18,2055,1459,828,282,862,0.709976


In [18]:
samples['full_reads'] = df_res.full_reads
samples.to_excel('results/samples_info.xlsx')